# Bronze — Ingestão fiel dos custos de produção da soja (CONAB)

Lê o `.xls` de origem (515 abas `[OGM-]Município-UF-Ano`), extrai cada linha de
custo no **formato-alvo** (8 colunas + rastreabilidade) e grava em
`bronze.custo_soja_raw` no Postgres.

**Filosofia Bronze:** captura fiel. Valores entram crus (tabs, produtividade como
texto, totais incluídos). Limpeza e normalização são responsabilidade da Silver.

Decisões desta camada (definidas com o dono do dado):
- **Todas as linhas** de dado: itens, subitens (`3.1`, `3.2`), pais e totais.
- **Categoria** (grupo romano I, II, III…) propagada para cada linha.
- **Produtividade** repetida em cada linha, **crua** (como veio na origem).
- `local`, `estado`, `ano`, `tipo_cultivo` extraídos do nome da aba.
- Colunas extras `aba_origem`, `linha_origem` para rastreabilidade.
- Aba `Índice` ignorada.
- **Idempotente:** recria a tabela do zero a cada execução.

## 1. Imports e configuração

In [2]:
import sys 
print(sys.executable)

c:\Users\cauai.Capozzoli\Desktop\projetos\repo-local\.venv\Scripts\python.exe


In [3]:
import re
from pathlib import Path

import xlrd
import pandas as pd
from sqlalchemy import text

# Conexão única do projeto (lê o .env)
from db import get_engine

# Caminho do arquivo de origem
ARQUIVO = Path("..") / "dados" / "raw" / "dados_soja.xls"
assert ARQUIVO.exists(), f"Arquivo não encontrado: {ARQUIVO.resolve()}"

TABELA = "bronze.custo_soja_raw"
print("Origem:", ARQUIVO.resolve())

Origem: C:\Users\cauai.Capozzoli\Desktop\projetos\repo-local\custos_producao\dados\raw\dados_soja.xls


## 2. Funções de parsing

O parser lida com as **3 eras de layout** do arquivo (4, 5–7 e 13–21 colunas) de
forma unificada. A chave é **não** confiar na posição fixa das colunas de valor:
nas eras largas (2011–2020) os valores estão em colunas não-adjacentes por causa
de células mescladas. A estratégia robusta é ler os **números na ordem** em que
aparecem após a descrição — o 1º número é R$/ha, o 2º é R$/60 kg — o que vale
igual nas três eras.

In [4]:
RE_NOME   = re.compile(r"^(OGM-)?(.*)-([A-Z]{2})-(\d{4})$")
RE_ROMANO = re.compile(r"^\s*(I|II|III|IV|V|VI|VII|VIII|IX|X)\s*-\s*", re.I)
RE_ITEM   = re.compile(r"^\s*\d+(\.\d+)?\s*-")   # "1 - ", "3.1 - " ...
RE_TOTAL  = re.compile(r"(TOTAL|CUSTO VAR|CUSTO OPER|CUSTO FIXO|CUSTO TOTAL)", re.I)


def parse_nome_aba(nome: str):
    """Extrai local/estado/ano/tipo_cultivo do nome '[OGM-]Município-UF-Ano'."""
    m = RE_NOME.match(nome.strip())
    if not m:
        return None
    ogm, mun, uf, ano = m.groups()
    return {
        "local": mun.strip(),
        "estado": uf,
        "ano": int(ano),
        "tipo_cultivo": "transgenico" if ogm else "convencional",
    }


def achar_cabecalho(sh):
    """Linha e coluna onde está 'DISCRIMINAÇÃO' (início da tabela de custos)."""
    for r in range(min(sh.nrows, 20)):
        for c in range(sh.ncols):
            if "DISCRIMINA" in str(sh.cell_value(r, c)).upper():
                return r, c
    return None, None


def achar_produtividade(sh):
    """Produtividade CRUA — número em célula própria ou texto embutido."""
    for r in range(min(sh.nrows, 20)):
        for c in range(sh.ncols):
            v = str(sh.cell_value(r, c))
            if "rodutividade" in v.lower():
                for cc in range(c + 1, sh.ncols):
                    nxt = sh.cell_value(r, cc)
                    if str(nxt).strip():
                        return str(nxt) if ":" not in v else f"{v}|{nxt}"
                return v
    return None


def nums_ordem(sh, r, col_desc):
    """Números reais (cell_type==2) após a descrição, na ordem das colunas."""
    return [sh.cell_value(r, c)
            for c in range(col_desc + 1, sh.ncols)
            if sh.cell_type(r, c) == 2]

### Parser de uma aba

Retorna a lista de linhas no formato-alvo e um status. Não aborta em erro: quem
chama decide o que fazer com abas problemáticas (registradas no relatório final).

In [5]:
def parse_aba(sh, nome: str):
    meta = parse_nome_aba(nome)
    if meta is None:
        return [], "nome fora do padrão"

    hdr_row, col_desc = achar_cabecalho(sh)
    if hdr_row is None:
        return [], "sem cabeçalho DISCRIMINAÇÃO"

    prod = achar_produtividade(sh)

    linhas, categoria = [], None
    for r in range(hdr_row + 1, sh.nrows):
        desc = sh.cell_value(r, col_desc)
        desc_s = str(desc).replace("\t", "").strip() if desc is not None else ""
        if not desc_s:
            continue

        # Grupo romano -> vira a categoria corrente e é propagado
        if RE_ROMANO.match(desc_s):
            categoria = str(desc).strip()
            continue

        # Só linhas de dado: item, subitem ou total (ignora rodapés/elaboração)
        if not (RE_ITEM.match(desc_s) or RE_TOTAL.search(desc_s)):
            continue

        nums = nums_ordem(sh, r, col_desc)
        linhas.append({
            "descricao_custo": str(desc),                 # CRU (com tabs)
            "categoria": categoria,                       # grupo romano propagado
            "(R$/ha)": nums[0] if len(nums) >= 1 else None,
            "(R$/60 kg)": nums[1] if len(nums) >= 2 else None,
            "produtividade_media(kg/ha)": prod,           # CRU
            "local": meta["local"],
            "estado": meta["estado"],
            "ano": meta["ano"],
            "tipo_cultivo": meta["tipo_cultivo"],
            "aba_origem": nome,
            "linha_origem": r,
        })
    return linhas, "ok"

## 3. Processar todas as abas

In [6]:
wb = xlrd.open_workbook(ARQUIVO)

todas, falhas, ok = [], [], 0
for nome in wb.sheet_names():
    if nome.strip().lower() == "índice":      # aba de navegação, não é custo
        continue
    linhas, status = parse_aba(wb.sheet_by_name(nome), nome)
    if status != "ok" or not linhas:
        falhas.append((nome, status, len(linhas)))
    else:
        ok += 1
        todas.extend(linhas)

df = pd.DataFrame(todas)
print(f"Abas processadas: {ok}")
print(f"Abas com problema: {len(falhas)}")
print(f"Linhas extraídas: {len(df):,}")
if falhas:
    print("\nAbas problemáticas:")
    for f in falhas:
        print("  ", f)

Abas processadas: 515
Abas com problema: 0
Linhas extraídas: 23,727


## 4. Conferência rápida

Amostra de uma aba de cada era, para inspeção visual antes de gravar.

In [7]:
for aba in ["Barreiras-BA-2006", "Barreiras-BA-2011", "Barreiras-BA-2021"]:
    sub = df[df.aba_origem == aba]
    print(f"\n=== {aba} ({len(sub)} linhas) ===")
    display(sub[["descricao_custo", "categoria", "(R$/ha)", "(R$/60 kg)"]].head(8))


=== Barreiras-BA-2006 (30 linhas) ===


,descricao_custo,categoria,(R$/ha),(R$/60 kg)
0,1 - Operação com aviões,I - DESPESAS DE CUSTEIO DA LAVOURA,0.00,0.00
1,2 - Operação com máquinas,I - DESPESAS DE CUSTEIO DA LAVOURA,143.41,2.99
2,3 - Mão-de-obra temporária,I - DESPESAS DE CUSTEIO DA LAVOURA,5.85,0.12
3,4 - Mão-de-obra fixa,I - DESPESAS DE CUSTEIO DA LAVOURA,21.00,0.44
4,5 - Sementes,I - DESPESAS DE CUSTEIO DA LAVOURA,40.50,0.84
5,6 - Fertilizantes,I - DESPESAS DE CUSTEIO DA LAVOURA,267.00,5.56
6,7 - Defensivos,I - DESPESAS DE CUSTEIO DA LAVOURA,212.25,4.42
7,TOTAL DAS DESPESAS DE CUSTEIO DA LAVOURA (A),I - DESPESAS DE CUSTEIO DA LAVOURA,690.01,14.37



=== Barreiras-BA-2011 (58 linhas) ===


,descricao_custo,categoria,(R$/ha),(R$/60 kg)
150,1 - Operação com animal,I - DESPESAS DE CUSTEIO DA LAVOURA,0.00,0.00
151,2 - Operação com Avião,I - DESPESAS DE CUSTEIO DA LAVOURA,0.00,0.00
152,3 - Operação com máquinas:,I - DESPESAS DE CUSTEIO DA LAVOURA,NaN,NaN
153,\t \t \t \t \t \t3.1 - Tratores e Colheitadeiras,I - DESPESAS DE CUSTEIO DA LAVOURA,148.31,3.08
154,\t \t \t \t \t \t3.2 - Conjunto de Irrigação,I - DESPESAS DE CUSTEIO DA LAVOURA,0.00,0.00
155,4 - Aluguel de Máquinas,I - DESPESAS DE CUSTEIO DA LAVOURA,0.00,0.00
156,5 - Aluguel de Animais,I - DESPESAS DE CUSTEIO DA LAVOURA,0.00,0.00
157,6 - Mão-de-obra,I - DESPESAS DE CUSTEIO DA LAVOURA,5.40,0.13



=== Barreiras-BA-2021 (48 linhas) ===


,descricao_custo,categoria,(R$/ha),(R$/60 kg)
701,1 - Operação com animal,I - DESPESAS DO CUSTEIO,0.00,0.00000
702,2 - Operação com Avião,I - DESPESAS DO CUSTEIO,0.00,0.00000
703,3 - Operação com máquinas:,I - DESPESAS DO CUSTEIO,NaN,NaN
704,\t \t \t \t \t \t3.1 - Tratores e Colheitadeiras,I - DESPESAS DO CUSTEIO,217.48,3.56525
705,\t \t \t \t \t \t3.2 - Conjunto de Irrigação,I - DESPESAS DO CUSTEIO,0.00,0.00000
706,4 - Aluguel de Máquinas,I - DESPESAS DO CUSTEIO,0.00,0.00000
707,5 - Aluguel de Animais,I - DESPESAS DO CUSTEIO,0.00,0.00000
708,6 - Mão de obra,I - DESPESAS DO CUSTEIO,5.99,0.09820


## 5. Gravar no Postgres (`bronze.custo_soja_raw`)

**Idempotente:** apaga e recria a tabela a cada execução, então rodar o notebook
duas vezes não duplica dados. O schema `bronze` já existe (criado na subida do
banco).

In [8]:
engine = get_engine()

# Idempotência: recria a tabela do zero
with engine.begin() as conn:
    conn.execute(text(f"DROP TABLE IF EXISTS {TABELA};"))

# Grava o DataFrame. Nomes de coluna com símbolos são preservados como estão.
df.to_sql(
    name="custo_soja_raw",
    schema="bronze",
    con=engine,
    if_exists="replace",
    index=False,
    chunksize=5000,
    method="multi",
)
print(f"Gravado em {TABELA}: {len(df):,} linhas")

Gravado em bronze.custo_soja_raw: 23,727 linhas


## 6. Validar no banco

Confirma a contagem e vê uma amostra direto do Postgres.

In [9]:
with engine.connect() as conn:
    total = conn.execute(text(f"SELECT COUNT(*) FROM {TABELA}")).scalar()
    n_abas = conn.execute(text(f"SELECT COUNT(DISTINCT aba_origem) FROM {TABELA}")).scalar()
    por_tipo = conn.execute(text(
        f"SELECT tipo_cultivo, COUNT(*) FROM {TABELA} GROUP BY tipo_cultivo"
    )).fetchall()

print(f"Total de linhas no banco: {total:,}")
print(f"Abas distintas: {n_abas}")
print("Por tipo de cultivo:", dict(por_tipo))

pd.read_sql(f"SELECT * FROM {TABELA} WHERE aba_origem = 'Barreiras-BA-2006' LIMIT 10", engine)

Total de linhas no banco: 23,727
Abas distintas: 515
Por tipo de cultivo: {'transgenico': 5728, 'convencional': 17999}


,descricao_custo,categoria,(R$/ha),(R$/60 kg),produtividade_media(kg/ha),local,estado,ano,tipo_cultivo,aba_origem,linha_origem
0,1 - Operação com aviões,I - DESPESAS DE CUSTEIO DA LAVOURA,0.00,0.00,Produtividade media: |2880.0,Barreiras,BA,2006,convencional,Barreiras-BA-2006,11
1,2 - Operação com máquinas,I - DESPESAS DE CUSTEIO DA LAVOURA,143.41,2.99,Produtividade media: |2880.0,Barreiras,BA,2006,convencional,Barreiras-BA-2006,12
2,3 - Mão-de-obra temporária,I - DESPESAS DE CUSTEIO DA LAVOURA,5.85,0.12,Produtividade media: |2880.0,Barreiras,BA,2006,convencional,Barreiras-BA-2006,13
3,4 - Mão-de-obra fixa,I - DESPESAS DE CUSTEIO DA LAVOURA,21.00,0.44,Produtividade media: |2880.0,Barreiras,BA,2006,convencional,Barreiras-BA-2006,14
4,5 - Sementes,I - DESPESAS DE CUSTEIO DA LAVOURA,40.50,0.84,Produtividade media: |2880.0,Barreiras,BA,2006,convencional,Barreiras-BA-2006,15
5,6 - Fertilizantes,I - DESPESAS DE CUSTEIO DA LAVOURA,267.00,5.56,Produtividade media: |2880.0,Barreiras,BA,2006,convencional,Barreiras-BA-2006,16
6,7 - Defensivos,I - DESPESAS DE CUSTEIO DA LAVOURA,212.25,4.42,Produtividade media: |2880.0,Barreiras,BA,2006,convencional,Barreiras-BA-2006,17
7,TOTAL DAS DESPESAS DE CUSTEIO DA LAVOURA (A),I - DESPESAS DE CUSTEIO DA LAVOURA,690.01,14.37,Produtividade media: |2880.0,Barreiras,BA,2006,convencional,Barreiras-BA-2006,18
8,1 - Transporte externo,II - DESPESAS PÓS-COLHEITA,23.04,0.48,Produtividade media: |2880.0,Barreiras,BA,2006,convencional,Barreiras-BA-2006,20
9,"2 - Recepção, limpeza, secagem, armazenagem ...",II - DESPESAS PÓS-COLHEITA,11.60,0.24,Produtividade media: |2880.0,Barreiras,BA,2006,convencional,Barreiras-BA-2006,21
